In [ ]:

# PROJECT 8: SUPPLY CHAIN DISRUPTION PREDICTION

# STEP 1: IMPORT LIBRARIES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    ConfusionMatrixDisplay
)

np.random.seed(42)


# STEP 2: CREATE SYNTHETIC SUPPLY CHAIN DATA

n = 1000

df = pd.DataFrame({
    "supplier_delay_history": np.random.poisson(2, n),
    "inventory_coverage_days": np.random.uniform(0, 40, n),
    "transit_days": np.random.uniform(1, 20, n),
    "weather_risk": np.random.binomial(1, 0.2, n),
    "supplier_quality_score": np.random.uniform(1, 5, n)
})

# Generate illustrative disruption probabilities
logit = (
    -2
    + 0.45 * df["supplier_delay_history"]
    - 0.06 * df["inventory_coverage_days"]
    + 0.09 * df["transit_days"]
    + 1.0 * df["weather_risk"]
    - 0.35 * df["supplier_quality_score"]
)

probability = 1 / (1 + np.exp(-logit))

# 0 = No disruption, 1 = Disruption
df["disruption"] = (
    np.random.rand(n) < probability
).astype(int)


# STEP 3: EXPLORE THE DATA

print("SAMPLE SUPPLY CHAIN DATA")
print(df.head(10))

print("\nDISRUPTION DISTRIBUTION")
print(df["disruption"].value_counts())


# STEP 4: PREPARE DATA FOR MODELING

X = df.drop(columns=["disruption"])
y = df["disruption"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


# STEP 5: TRAIN RANDOM FOREST MODEL

model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]


# STEP 6: EVALUATE THE MODEL

print("\nMODEL PERFORMANCE")

print(
    "Accuracy:",
    round(accuracy_score(y_test, predictions), 3)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, probabilities), 3)
)

print("\nCLASSIFICATION REPORT")

print(classification_report(
    y_test,
    predictions,
    zero_division=0
))


# STEP 7: DISPLAY CONFUSION MATRIX

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=["No Disruption", "Disruption"]
)

plt.title("Supply Chain Disruption Prediction")
plt.tight_layout()
plt.show()


# STEP 8: ANALYZE IMPORTANT FEATURES

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nFEATURE IMPORTANCE")
print(importance.round(3))

importance.sort_values().plot(
    kind="barh",
    figsize=(8, 5)
)

plt.title("Factors Used by the Disruption Model")
plt.xlabel("Feature Importance")
plt.tight_layout()
plt.show()


# STEP 9: ESTIMATE DISRUPTION RISK

df["disruption_risk"] = model.predict_proba(X)[:, 1]

df["risk_category"] = np.select(
    [
        df["disruption_risk"] >= 0.65,
        df["disruption_risk"] >= 0.40
    ],
    [
        "High Risk",
        "Medium Risk"
    ],
    default="Low Risk"
)

df["recommended_action"] = np.select(
    [
        df["risk_category"] == "High Risk",
        df["risk_category"] == "Medium Risk"
    ],
    [
        "Review shipment and prepare contingency plan",
        "Monitor supplier and inventory status"
    ],
    default="Continue routine monitoring"
)

print("\nHIGHEST-RISK SUPPLY CHAIN RECORDS")

print(
    df.sort_values(
        "disruption_risk",
        ascending=False
    ).head(15).round(3).to_string(index=False)
)


# STEP 10: BUSINESS RECOMMENDATIONS

print("\nBUSINESS RECOMMENDATIONS")

print("1. Monitor suppliers with repeated delivery delays.")
print("2. Review inventory coverage for critical products.")
print("3. Prepare contingency plans for weather-related risks.")
print("4. Evaluate alternative suppliers for critical materials.")
print("5. Track actual disruption outcomes to improve the model.")

print("\nPROJECT 8 COMPLETED SUCCESSFULLY")
print("Note: Synthetic data is used for demonstration.")
